# Web APIs: Building REST Endpoints with Flask & FastAPI

Build HTTP APIs for web services, microservices, and third-party integrations.

**Install:** `pip install flask fastapi uvicorn`

## 1. Flask Basics: Simple API

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

# In-memory data store
users = [
    {"id": 1, "name": "Alice", "email": "alice@example.com"},
    {"id": 2, "name": "Bob", "email": "bob@example.com"}
]

# GET: Retrieve all users
@app.route('/api/users', methods=['GET'])
def get_users():
    return jsonify(users)
    # HTTP GET /api/users -> [{"id": 1, "name": "Alice", ...}]

# GET: Retrieve one user by ID
@app.route('/api/users/<int:user_id>', methods=['GET'])
def get_user(user_id):
    user = next((u for u in users if u['id'] == user_id), None)
    if not user:
        return jsonify({"error": "User not found"}), 404
    return jsonify(user)
    # HTTP GET /api/users/1 -> {"id": 1, "name": "Alice", ...}

# POST: Create new user
@app.route('/api/users', methods=['POST'])
def create_user():
    data = request.json
    new_user = {
        "id": max([u['id'] for u in users]) + 1,
        "name": data.get('name'),
        "email": data.get('email')
    }
    users.append(new_user)
    return jsonify(new_user), 201
    # HTTP POST /api/users with {"name": "Charlie", "email": "charlie@example.com"}
    # Response: {"id": 3, "name": "Charlie", ...}, status 201 Created

# PUT: Update user
@app.route('/api/users/<int:user_id>', methods=['PUT'])
def update_user(user_id):
    user = next((u for u in users if u['id'] == user_id), None)
    if not user:
        return jsonify({"error": "User not found"}), 404
    data = request.json
    user.update(data)
    return jsonify(user)

# DELETE: Remove user
@app.route('/api/users/<int:user_id>', methods=['DELETE'])
def delete_user(user_id):
    global users
    users = [u for u in users if u['id'] != user_id]
    return '', 204
    # HTTP DELETE /api/users/1 -> status 204 No Content

if __name__ == '__main__':
    app.run(debug=True, port=5000)  # Start at http://localhost:5000
```

## 2. FastAPI: Modern & Async

```python
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

# Request/Response model (automatic validation & docs)
class User(BaseModel):
    name: str
    email: str
    age: Optional[int] = None

# In-memory data
users = {
    1: {"id": 1, "name": "Alice", "email": "alice@example.com"},
    2: {"id": 2, "name": "Bob", "email": "bob@example.com"}
}

# GET all
@app.get("/api/users")
async def get_users():
    return list(users.values())

# GET one
@app.get("/api/users/{user_id}")
async def get_user(user_id: int):
    if user_id not in users:
        raise HTTPException(status_code=404, detail="User not found")
    return users[user_id]

# POST (create)
@app.post("/api/users", status_code=201)
async def create_user(user: User):
    new_id = max(users.keys()) + 1 if users else 1
    new_user = {"id": new_id, **user.dict()}
    users[new_id] = new_user
    return new_user

# PUT (update)
@app.put("/api/users/{user_id}")
async def update_user(user_id: int, user: User):
    if user_id not in users:
        raise HTTPException(status_code=404, detail="User not found")
    users[user_id].update(user.dict())
    return users[user_id]

# DELETE
@app.delete("/api/users/{user_id}", status_code=204)
async def delete_user(user_id: int):
    if user_id not in users:
        raise HTTPException(status_code=404, detail="User not found")
    del users[user_id]

# Run: uvicorn main:app --reload
# Docs: http://localhost:8000/docs (automatic Swagger UI)
```

## 3. Request Validation (FastAPI/Pydantic)

Automatically validate request data.

```python
from pydantic import BaseModel, EmailStr, Field
from typing import Optional

class UserCreate(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    email: EmailStr  # Validates email format
    age: Optional[int] = Field(None, ge=0, le=150)  # 0-150 years old

# If request has:
# {"name": "Alice", "email": "not-an-email", "age": 25}
# FastAPI returns 422 Unprocessable Entity with error details

# If request has:
# {"name": "Alice", "email": "alice@example.com", "age": 25}
# FastAPI parses and passes as UserCreate instance
```

## 4. Authentication & Authorization

Secure endpoints with token-based auth.

```python
from fastapi import Depends, HTTPException, status
from fastapi.security import HTTPBearer

security = HTTPBearer()

# Token validation dependency
async def verify_token(credentials = Depends(security)):
    token = credentials.credentials if hasattr(credentials, "credentials") else credentials
    if token != "secret-token-123":
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid token"
        )
    return token

# Protect endpoint with Depends
@app.get("/api/protected")
async def protected_route(token: str = Depends(verify_token)):
    return {"message": "Access granted", "token": token}

# Usage:
# curl -H "Authorization: Bearer secret-token-123" http://localhost:8000/api/protected
# Response: {"message": "Access granted", "token": "secret-token-123"}
```

## 5. Error Handling

Return meaningful error responses.

```python
# HTTP Status Codes
# 200 OK - Success
# 201 Created - Resource created
# 204 No Content - Success, no response body
# 400 Bad Request - Client error (invalid data)
# 401 Unauthorized - Missing/invalid auth
# 403 Forbidden - Authenticated but no permission
# 404 Not Found - Resource doesn't exist
# 500 Internal Server Error - Server error

# Error handling in FastAPI
@app.get("/api/users/{user_id}")
async def get_user(user_id: int):
    try:
        if user_id < 1:
            raise HTTPException(status_code=400, detail="ID must be positive")
        user = users.get(user_id)
        if not user:
            raise HTTPException(status_code=404, detail="User not found")
        return user
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
```

## 6. Testing APIs in Jupyter

```python
# Test Flask API
from flask.testing import FlaskClient

with app.test_client() as client:
    # GET
    response = client.get("/api/users")
    assert response.status_code == 200
    data = response.json
    print(f"Users: {data}")
    
    # POST
    response = client.post(
        "/api/users",
        json={"name": "Charlie", "email": "charlie@example.com"}
    )
    assert response.status_code == 201
    print(f"Created: {response.json}")

# Test FastAPI
from fastapi.testclient import TestClient

client = TestClient(app)

# GET
response = client.get("/api/users")
assert response.status_code == 200
print(response.json())

# POST with validation
response = client.post(
    "/api/users",
    json={"name": "David", "email": "david@example.com"}
)
print(response.json())
```

---

**Quick Comparison:**

| Feature | Flask | FastAPI |
|---------|-------|----------|
| Framework | Lightweight | Full-featured |
| Async | With async_to_sync | Native |
| Validation | Manual | Automatic (Pydantic) |
| Documentation | No docs | Auto Swagger/OpenAPI |
| Performance | Good | Excellent |
| Learning curve | Easy | Moderate |

**API Pattern Summary:**
1. Define models (Pydantic)
2. Create endpoints (Flask routes / FastAPI decorators)
3. Handle requests/responses (JSON, validation)
4. Add auth (tokens, API keys)
5. Test with client (requests or TestClient)
6. Deploy (Heroku, AWS, Docker)